# Air Quality Data Analysis with Python
## Notebook 6 · Bonus: Live Data from the OpenAQ API

⏱️ About 40 minutes &nbsp;·&nbsp; ⬅️ Builds on Notebooks 2–5 &nbsp;·&nbsp; 🎁 Optional

So far the data came as CSV files we prepared for you. This bonus notebook closes
the loop: you'll fetch **live measurements** yourself from
[OpenAQ](https://openaq.org), the open platform that aggregates air-quality data
worldwide (it's where this course's data originally came from).

Two things make this notebook different:

* ⚠️ **It needs a free API key.** Register at
  <https://explore.openaq.org/register> (takes two minutes), then find your key on
  your account page. Without a key the requests below will return errors —
  everything else in the course works without it.
* You'll meet **JSON**, the format web APIs speak — and discover it's exactly the
  dictionaries-and-lists you learned in Notebook 2.

### 1. Keys are secrets

An API key identifies *you*. Treat it like a password: **never paste it into a
code cell** — anyone you share the notebook with (including your instructor!)
would see it. The cell below uses `getpass`, which prompts for the key and keeps
it out of the notebook.

(Colab also has a nicer built-in: the 🔑 **Secrets** panel on the left. Store the
key once under the name `OPENAQ_API_KEY`, enable "notebook access", and this cell
will find it automatically.)

In [ ]:
import os
from getpass import getpass

API_KEY = os.environ.get("OPENAQ_API_KEY", "")
if not API_KEY:
    try:  # Colab Secrets, if you stored the key there
        from google.colab import userdata
        API_KEY = userdata.get("OPENAQ_API_KEY")
    except Exception:
        pass
if not API_KEY:
    API_KEY = getpass("Paste your OpenAQ API key (input stays hidden): ")
print("Key loaded:", "yes" if API_KEY else "no — requests below will fail")

### 2. Your first API request

The `requests` library fetches URLs. An API request has three parts: the
**endpoint** (what you're asking for), the **parameters** (details of the ask),
and **headers** (here: your key). This asks OpenAQ for PM2.5 monitoring locations
within 12 km of central Lagos:

In [ ]:
import requests

response = requests.get(
    "https://api.openaq.org/v3/locations",
    params={
        "coordinates": "6.5244,3.3792",  # latitude,longitude of Lagos
        "radius": 12000,                 # metres
        "parameters_id": 2,              # 2 = PM2.5 in OpenAQ's catalogue
        "limit": 10,
    },
    headers={"X-API-Key": API_KEY},
    timeout=30,
)
print("Status code:", response.status_code)  # 200 means success

(Common non-200 answers: **401** = missing/wrong key, **429** = too many requests,
slow down.)

### 3. JSON is Notebook 2 in disguise

The response body is **JSON** — and `.json()` turns it into… Python dictionaries
and lists. Everything you learned in Notebook 2 applies directly:

In [ ]:
payload = response.json()
print(type(payload))
print(payload.keys())

A dict! Its `"results"` key holds a **list** of locations, each a **dict**. Drill
in with exactly the indexing you already know:

In [ ]:
results = payload["results"]
first = results[0]
print("Locations returned:", len(results))
print("First location:", first["name"])
print("Its coordinates:", first["coordinates"])

### 4. From JSON records to a DataFrame

An API's list-of-dicts is the *records* shape from Notebook 2 — and
`pd.DataFrame(...)` accepts it directly. Usually you first distil each record down
to the fields you care about with a loop:

In [ ]:
import pandas as pd

records = []
for loc in results:
    records.append({
        "id": loc["id"],
        "name": loc["name"],
        "provider": loc["provider"]["name"],
        "last_seen": loc["datetimeLast"]["utc"] if loc["datetimeLast"] else None,
    })

locations = pd.DataFrame(records)
locations

Run the checker cell, then it's your turn:

In [ ]:
def check(name, test, hint=""):
    """Run test() and print a friendly ✅ or 💡 — never an error message."""
    try:
        ok = bool(test())
    except Exception:
        ok = False
    if ok:
        print(f"✅ {name} — looks right!")
    else:
        print(f"💡 {name} — not quite yet. Hint: {hint}")

✏️ **Your turn 6.1** — Ask the same endpoint for PM2.5 locations around **Abuja**
(coordinates `9.0765,7.3986`, radius 25000 m) and build a DataFrame
`abuja_locations` with the columns `id` and `name`.

In [ ]:
# reuse the request pattern from section 2, changing coordinates/radius/limit,
# then distil results into records and build the DataFrame
abuja_response = ...
abuja_locations = ...

In [ ]:
check("abuja_locations is a non-empty table",
      lambda: len(abuja_locations) > 0 and "name" in abuja_locations.columns,
      "build a list of {'id': ..., 'name': ...} dicts from the results, then pd.DataFrame(...)")

### 5. Recent hourly values for a known sensor

Each location carries one or more **sensors**; historical values hang off the
sensor. Our Oshodi Bus Terminal PM2.5 sensor has id `13573593`. The `/hours`
endpoint returns hourly aggregates — here, the last 14 days:

In [ ]:
sensor_id = 13573593  # PM2.5 sensor at Oshodi Bus Terminal (location 5038498)

two_weeks_ago = (pd.Timestamp.now(tz="UTC") - pd.Timedelta(days=14)).isoformat()

hours_response = requests.get(
    f"https://api.openaq.org/v3/sensors/{sensor_id}/hours",
    params={"datetime_from": two_weeks_ago, "limit": 1000},
    headers={"X-API-Key": API_KEY},
    timeout=60,
)
hourly_results = hours_response.json()["results"]
print(f"Hourly records returned: {len(hourly_results)}")
if hourly_results:
    print("One record:", {k: hourly_results[-1][k] for k in ("value", "period")})

Each record's timestamp sits inside `period → datetimeFrom → utc`. Distil, frame,
and apply the Notebook 3 datetime recipe:

In [ ]:
live = pd.DataFrame({
    "datetime": [r["period"]["datetimeFrom"]["utc"] for r in hourly_results],
    "pm25_value": [r["value"] for r in hourly_results],
})
live["datetime"] = pd.to_datetime(live["datetime"], utc=True)
live = live.set_index("datetime").tz_convert("Africa/Lagos").sort_index()
print(f"{len(live)} hours, {live.index.min()} → {live.index.max()}")

And plot it in the house style — live data from a sensor on the other side of the
world, in four lines of pandas:

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (9.5, 4.2),
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y",
    "grid.color": "#cbcbcb", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlelocation": "left",
})

ax = live["pm25_value"].plot(color="#0072B2", linewidth=1.5)
ax.set_ylabel("PM2.5 (µg/m³)")
ax.set_xlabel("")
ax.set_ylim(0, None)
ax.tick_params(axis="x", rotation=0)
ax.set_title("Oshodi Bus Terminal, the last two weeks — fetched live from OpenAQ");

### 6. Where to go from here

* **Bulk history without an API key**: OpenAQ also publishes its full archive as
  daily CSV files on a public server — that's how this course's datasets were
  built. See `scripts/prepare_data.py` in the course repository for a working,
  reusable downloader.
* **The AQ agent**: everything you've built by hand across these six notebooks —
  loading, cleaning, QA checks, diurnal and monthly charts, multi-site
  comparisons — the AQ agent's Data Explorer does conversationally, on the same
  OpenAQ data plus your own uploads. You now know exactly what it's doing under
  the hood, and how to check its work.

### 7. Recap — and congratulations

* APIs answer structured requests; **keys are secrets** (getpass / Colab Secrets).
* JSON **is** dicts-and-lists; `results` → records → `pd.DataFrame`.
* The `/locations` endpoint finds stations; `/sensors/{id}/hours` returns hourly
  history; the datetime recipe applies unchanged.

🎓 **That's the course.** You started with `print("Welcome!")`; you're ending by
pulling live measurements from a global network and turning them into
publication-quality analysis. Well done — and welcome to air-quality data science.